# SGLang 流式响应输出 — 实时获取推理结果

**定位：** 演示如何使用 SGLang 的流式（Streaming / SSE）输出功能，实现逐 token 实时打印，测量首 token 延迟等关键指标，并封装可复用的工具函数。

```bash
# 默认启动命令
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

> 本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台

---

## 硬件与环境要求

| 项目 | 最低要求 | 推荐配置 |
|------|---------|----------|
| GPU | NVIDIA GPU（算力 ≥ 7.0） | RTX 3060 及以上 |
| 显存 (VRAM) | ≥ 6 GB | ≥ 8 GB |
| 驱动版本 | ≥ 525.0 | 最新稳定版 |
| CUDA 版本 | ≥ 12.1 | 12.4+ |
| Python | 3.9 – 3.12 | 3.10 |
| 操作系统 | Linux / WSL2 | Ubuntu 22.04 |

---

## 依赖安装

推荐使用虚拟环境隔离依赖：

```bash
# 创建并激活虚拟环境
python -m venv sglang-env
source sglang-env/bin/activate

# 安装 SGLang 全部组件及所需依赖
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和 Python 环境正常
2. 如需安装依赖，运行「可选安装」单元格
3. 运行「启动 SGLang 服务」单元格，等待服务就绪
4. 依次运行流式与非流式调用的对比实验
5. 观察流式输出时 token 逐个打印的效果
6. 完成后运行「清理资源」单元格释放 GPU 显存

---

In [ ]:
# === 环境检查 ===
import subprocess  # 导入子进程模块
import sys  # 导入系统模块

print('=' * 50)  # 打印分隔线
print('SGLang 流式输出 — 环境检查')  # 打印标题
print('=' * 50)  # 打印分隔线

print(f'Python 版本: {sys.version}')  # 输出当前 Python 版本

try:  # 尝试执行 nvidia-smi 命令
    result = subprocess.run(  # 运行 nvidia-smi 查询 GPU 信息
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],  # 查询 GPU 名称和显存
        capture_output=True, text=True, timeout=10  # 捕获输出并设置超时
    )  # 命令执行完成
    if result.returncode == 0:  # 如果命令执行成功
        print(f'GPU 信息: {result.stdout.strip()}')  # 输出 GPU 信息
    else:  # 如果命令执行失败
        print('警告: nvidia-smi 执行失败')  # 输出警告
except FileNotFoundError:  # 如果找不到 nvidia-smi
    print('警告: 未找到 nvidia-smi，请确认已安装 NVIDIA 驱动')  # 提示用户

try:  # 尝试导入 sglang
    import sglang  # 导入 sglang 模块
    print(f'SGLang 版本: {sglang.__version__}')  # 输出版本号
except ImportError:  # 如果未安装
    print('SGLang 未安装，请运行下方安装单元格')  # 提示安装

try:  # 尝试导入 openai
    import openai  # 导入 openai 模块
    print(f'OpenAI SDK 版本: {openai.__version__}')  # 输出版本号
except ImportError:  # 如果未安装
    print('OpenAI SDK 未安装')  # 提示安装

print('=' * 50)  # 打印分隔线

In [ ]:
# === 可选：安装依赖（已安装可跳过） ===
# import subprocess  # 导入子进程模块
# import sys  # 导入系统模块
# subprocess.check_call([  # 执行 pip 安装命令
#     sys.executable, '-m', 'pip', 'install', '-q',  # 使用当前解释器静默安装
#     'sglang[all]', 'openai>=1.0.0', 'requests>=2.28.0'  # 安装所需依赖
# ])  # 安装命令执行完成
# print('依赖安装完成')  # 打印安装成功提示

## 什么是流式输出？

在大模型推理中，生成一段文本需要逐 token 计算。传统的**非流式**调用会等所有 token 生成完毕后一次性返回，用户需要等待较长时间才能看到结果。

**流式输出（Streaming）** 则是一边生成一边返回，用户可以实时看到每个 token 的输出。底层基于 **SSE（Server-Sent Events）** 协议：

- 服务器通过 HTTP 长连接持续推送数据
- 每个数据块以 `data: {...}` 格式发送
- 最后以 `data: [DONE]` 标识流结束

**流式 vs 非流式对比：**

| 特性 | 非流式 | 流式 |
|------|--------|------|
| 用户体感 | 长时间等待后突然出现完整结果 | 逐字显示，体验流畅 |
| 首 token 延迟 | 等于总延迟 | 通常很短（毫秒级） |
| 总生成时间 | 相同 | 相同 |
| 实现复杂度 | 简单 | 略复杂（需处理数据流） |
| 适用场景 | 后台批处理、API 集成 | 聊天对话、实时交互 |

In [ ]:
# === 启动 SGLang 服务 ===
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求模块
import sys  # 导入系统模块

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'  # 定义使用的模型 ID
HOST = '127.0.0.1'  # 服务监听地址
PORT = 30000  # 服务监听端口

launch_cmd = [  # 构建启动命令列表
    sys.executable, '-m', 'sglang.launch_server',  # 调用 sglang 启动模块
    '--model-path', MODEL_ID,  # 指定模型路径
    '--host', HOST,  # 指定监听地址
    '--port', str(PORT),  # 指定端口号
]  # 命令构建完成

print(f'启动命令: {" ".join(launch_cmd)}')  # 打印启动命令

server_process = subprocess.Popen(  # 以子进程方式启动服务
    launch_cmd,  # 传入命令列表
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE,  # 捕获标准错误输出
)  # 子进程创建完成
print(f'服务进程已启动，PID = {server_process.pid}')  # 打印进程 ID

max_wait = 300  # 最大等待时间 300 秒
start_time = time.time()  # 记录开始时间
ready = False  # 服务就绪标志

while time.time() - start_time < max_wait:  # 在超时范围内持续轮询
    try:  # 尝试连接服务
        resp = requests.get(f'http://{HOST}:{PORT}/health', timeout=5)  # 发送健康检查请求
        if resp.status_code == 200:  # 如果返回 200
            elapsed = time.time() - start_time  # 计算已用时间
            print(f'服务启动成功！耗时 {elapsed:.1f} 秒')  # 打印成功信息
            ready = True  # 标记为就绪
            break  # 退出循环
    except requests.exceptions.ConnectionError:  # 连接失败
        pass  # 服务尚未就绪，继续等待
    time.sleep(2)  # 每 2 秒检查一次

if not ready:  # 如果超时
    print(f'服务未在 {max_wait} 秒内启动，请检查日志')  # 打印超时警告

In [ ]:
# === 非流式调用（对比基线） ===
from openai import OpenAI  # 导入 OpenAI 客户端
import time  # 导入时间模块

client = OpenAI(  # 创建 OpenAI 兼容客户端
    base_url='http://127.0.0.1:30000/v1',  # 指定 SGLang 服务地址
    api_key='EMPTY',  # API 密钥（SGLang 不需要真实密钥）
)  # 客户端创建完成

messages = [  # 构建对话消息列表
    {"role": "system", "content": "你是一个有帮助的助手。"},  # 系统提示
    {"role": "user", "content": "请用 200 字左右介绍一下人工智能的发展历史。"},  # 用户问题
]  # 消息构建完成

print('=== 非流式调用（对比基线） ===')  # 打印标题
print('等待完整响应...\n')  # 提示等待

start_time = time.time()  # 记录请求开始时间
response = client.chat.completions.create(  # 发送非流式聊天补全请求
    model='Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型
    messages=messages,  # 传入消息列表
    max_tokens=256,  # 最大输出 token 数
    stream=False,  # 关闭流式输出（默认值）
)  # 请求完成
total_time = time.time() - start_time  # 计算总耗时

print(f'模型回复:\n{response.choices[0].message.content}')  # 打印模型完整回复
print(f'\n总耗时: {total_time:.2f} 秒')  # 打印总耗时
print(f'输出 token 数: {response.usage.completion_tokens}')  # 打印输出 token 数
print('\n注意：非流式调用时，用户必须等待全部 token 生成完毕才能看到结果')  # 说明非流式特点

In [ ]:
# === 流式调用：使用 OpenAI SDK ===
from openai import OpenAI  # 导入 OpenAI 客户端
import time  # 导入时间模块

client = OpenAI(  # 创建 OpenAI 兼容客户端
    base_url='http://127.0.0.1:30000/v1',  # 指定服务地址
    api_key='EMPTY',  # API 密钥
)  # 客户端创建完成

messages = [  # 构建消息列表
    {"role": "system", "content": "你是一个有帮助的助手。"},  # 系统提示
    {"role": "user", "content": "请用 200 字左右介绍一下人工智能的发展历史。"},  # 用户问题
]  # 消息构建完成

print('=== 流式调用（OpenAI SDK） ===')  # 打印标题
print('实时输出每个 token：\n')  # 提示实时输出

start_time = time.time()  # 记录开始时间
first_token_time = None  # 首 token 时间初始化为 None
full_text = ''  # 完整文本初始化为空字符串

stream = client.chat.completions.create(  # 发送流式聊天补全请求
    model='Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型
    messages=messages,  # 传入消息列表
    max_tokens=256,  # 最大输出 token 数
    stream=True,  # 开启流式输出
)  # 流式请求创建完成

for chunk in stream:  # 遍历流中的每个数据块
    if chunk.choices[0].delta.content:  # 如果数据块包含文本内容
        token = chunk.choices[0].delta.content  # 提取本次的 token 文本
        if first_token_time is None:  # 如果这是收到的第一个 token
            first_token_time = time.time() - start_time  # 记录首 token 延迟
        full_text += token  # 将 token 累加到完整文本
        print(token, end='', flush=True)  # 实时打印 token，不换行

total_time = time.time() - start_time  # 计算总耗时
print(f'\n\n--- 统计信息 ---')  # 打印统计标题
print(f'首 token 延迟 (TTFT): {first_token_time:.3f} 秒')  # 打印首 token 延迟
print(f'总耗时: {total_time:.2f} 秒')  # 打印总耗时
print(f'生成字符数: {len(full_text)}')  # 打印生成的字符总数

In [ ]:
# === 流式调用：使用 requests + SSE 手动解析 ===
import requests  # 导入 HTTP 请求模块
import json  # 导入 JSON 模块
import time  # 导入时间模块

url = 'http://127.0.0.1:30000/v1/chat/completions'  # 定义请求 URL
headers = {'Content-Type': 'application/json'}  # 设置请求头

payload = {  # 构建请求体
    'model': 'Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型名称
    'messages': [  # 消息列表
        {'role': 'user', 'content': '什么是机器学习？请简要说明。'}  # 用户问题
    ],  # 消息列表结束
    'max_tokens': 200,  # 最大输出 token 数
    'stream': True,  # 开启流式输出
}  # 请求体构建完成

print('=== 使用 requests 手动解析 SSE ===')  # 打印标题
print('展示原始 SSE 数据格式：\n')  # 提示展示原始格式

start_time = time.time()  # 记录开始时间
full_text = ''  # 完整文本初始化
chunk_count = 0  # 数据块计数器

resp = requests.post(url, json=payload, headers=headers, stream=True)  # 发送流式 POST 请求

for line in resp.iter_lines(decode_unicode=True):  # 逐行读取响应数据
    if not line:  # 如果是空行
        continue  # 跳过空行
    if not line.startswith('data: '):  # 如果不是 SSE 数据行
        continue  # 跳过非数据行
    data_str = line[len('data: '):]  # 提取 data: 后面的 JSON 内容
    if data_str.strip() == '[DONE]':  # 如果收到结束标志
        print('\n\n[收到 DONE 信号，流式传输结束]')  # 打印结束信息
        break  # 退出循环
    try:  # 尝试解析 JSON 数据
        data = json.loads(data_str)  # 将字符串解析为 JSON 对象
        delta = data['choices'][0]['delta']  # 获取 delta 增量字段
        if 'content' in delta:  # 如果 delta 中包含 content
            token = delta['content']  # 提取 token 文本
            full_text += token  # 累加到完整文本
            chunk_count += 1  # 计数器加一
            if chunk_count <= 5:  # 前 5 个 chunk 展示原始 SSE 格式
                print(f'  SSE 原始数据: {line}')  # 打印原始 SSE 行
            elif chunk_count == 6:  # 第 6 个 chunk 时给出省略提示
                print('  ... (后续数据省略，直接显示最终结果) ...')  # 省略提示
    except json.JSONDecodeError:  # 如果 JSON 解析失败
        pass  # 忽略解析错误

total_time = time.time() - start_time  # 计算总耗时
print(f'\n完整回复: {full_text}')  # 打印完整回复内容
print(f'总数据块数: {chunk_count}')  # 打印接收到的数据块总数
print(f'总耗时: {total_time:.2f} 秒')  # 打印总耗时

In [ ]:
# === 流式输出 + 详细时间统计 ===
from openai import OpenAI  # 导入 OpenAI 客户端
import time  # 导入时间模块

client = OpenAI(  # 创建 OpenAI 兼容客户端
    base_url='http://127.0.0.1:30000/v1',  # 指定服务地址
    api_key='EMPTY',  # API 密钥
)  # 客户端创建完成

messages = [  # 构建消息列表
    {"role": "user", "content": "请列举 5 个中国古代四大发明并简要说明每个发明的重要性。"}  # 用户问题
]  # 消息构建完成

print('=== 流式输出 + 详细时间统计 ===\n')  # 打印标题

start_time = time.time()  # 记录总开始时间
first_token_time = None  # 首 token 时间初始化
token_times = []  # 每个 token 的到达时间戳列表
full_text = ''  # 完整文本初始化

stream = client.chat.completions.create(  # 发送流式请求
    model='Qwen/Qwen2.5-0.5B-Instruct',  # 指定模型
    messages=messages,  # 传入消息
    max_tokens=300,  # 最大输出 token 数
    stream=True,  # 开启流式输出
)  # 流式请求创建完成

for chunk in stream:  # 遍历流中的每个数据块
    if chunk.choices[0].delta.content:  # 如果数据块包含内容
        now = time.time()  # 记录当前时间戳
        token = chunk.choices[0].delta.content  # 提取 token 文本
        if first_token_time is None:  # 如果是第一个 token
            first_token_time = now - start_time  # 计算首 token 延迟
        token_times.append(now)  # 将时间戳添加到列表
        full_text += token  # 累加到完整文本
        print(token, end='', flush=True)  # 实时打印 token

total_time = time.time() - start_time  # 计算总耗时

inter_token_latencies = []  # 存储相邻 token 间的延迟
for i in range(1, len(token_times)):  # 遍历相邻 token 对
    latency = token_times[i] - token_times[i - 1]  # 计算两个 token 之间的间隔
    inter_token_latencies.append(latency)  # 添加到延迟列表

print(f'\n\n{"=" * 40}')  # 打印分隔线
print('详细时间统计：')  # 打印统计标题
print(f'  首 token 延迟 (TTFT):  {first_token_time:.3f} 秒')  # 打印首 token 延迟
print(f'  总生成时间:            {total_time:.2f} 秒')  # 打印总生成时间
print(f'  生成 token 数:         {len(token_times)}')  # 打印 token 总数

if inter_token_latencies:  # 如果有 token 间延迟数据
    avg_itl = sum(inter_token_latencies) / len(inter_token_latencies)  # 计算平均 token 间延迟
    min_itl = min(inter_token_latencies)  # 计算最小 token 间延迟
    max_itl = max(inter_token_latencies)  # 计算最大 token 间延迟
    print(f'  平均 token 间延迟:     {avg_itl * 1000:.1f} ms')  # 打印平均延迟（毫秒）
    print(f'  最小 token 间延迟:     {min_itl * 1000:.1f} ms')  # 打印最小延迟（毫秒）
    print(f'  最大 token 间延迟:     {max_itl * 1000:.1f} ms')  # 打印最大延迟（毫秒）
    gen_duration = total_time - first_token_time  # 计算纯生成阶段时长
    tps = len(token_times) / gen_duration if gen_duration > 0 else 0  # 计算每秒生成 token 数
    print(f'  生成速度:              {tps:.1f} tokens/秒')  # 打印生成速度

In [ ]:
# === 封装流式输出工具函数 ===
from openai import OpenAI  # 导入 OpenAI 客户端
import time  # 导入时间模块


def stream_chat(  # 定义可复用的流式聊天工具函数
    prompt,  # 用户输入提示文本（必填）
    model='Qwen/Qwen2.5-0.5B-Instruct',  # 模型名称（默认 Qwen2.5-0.5B）
    base_url='http://127.0.0.1:30000/v1',  # 服务地址（默认本地）
    max_tokens=256,  # 最大输出 token 数（默认 256）
    print_live=True,  # 是否实时打印输出（默认开启）
    collect_stats=True,  # 是否收集性能统计信息（默认开启）
):  # 函数参数定义完成
    """流式聊天工具函数：支持实时打印和性能统计"""  # 函数文档字符串
    client = OpenAI(base_url=base_url, api_key='EMPTY')  # 创建 OpenAI 客户端
    messages = [{"role": "user", "content": prompt}]  # 构建消息列表

    start_time = time.time()  # 记录开始时间
    first_token_time = None  # 首 token 时间初始化
    full_text = ''  # 完整文本初始化
    token_count = 0  # token 计数器

    stream = client.chat.completions.create(  # 创建流式请求
        model=model,  # 指定模型
        messages=messages,  # 传入消息
        max_tokens=max_tokens,  # 设置最大 token 数
        stream=True,  # 开启流式输出
    )  # 流式请求创建完成

    for chunk in stream:  # 遍历流中的每个数据块
        if chunk.choices[0].delta.content:  # 如果数据块包含内容
            token = chunk.choices[0].delta.content  # 提取 token 文本
            if first_token_time is None:  # 如果是第一个 token
                first_token_time = time.time() - start_time  # 记录首 token 延迟
            full_text += token  # 累加到完整文本
            token_count += 1  # token 计数加一
            if print_live:  # 如果开启了实时打印
                print(token, end='', flush=True)  # 实时打印 token

    total_time = time.time() - start_time  # 计算总耗时

    if print_live:  # 如果之前有实时打印
        print()  # 打印换行

    result = {'text': full_text, 'total_time': total_time}  # 构建基础返回结果

    if collect_stats and first_token_time is not None:  # 如果开启统计且有数据
        result['first_token_time'] = first_token_time  # 添加首 token 延迟
        result['token_count'] = token_count  # 添加 token 总数
        gen_duration = total_time - first_token_time  # 计算纯生成时长
        tps = token_count / gen_duration if gen_duration > 0 else 0  # 计算生成速度
        result['tokens_per_second'] = tps  # 添加生成速度

    return result  # 返回包含文本和统计信息的字典


# 测试工具函数
print('=== 测试 stream_chat 工具函数 ===\n')  # 打印测试标题

result = stream_chat(  # 调用流式聊天工具函数
    prompt='用一句话解释什么是深度学习。',  # 输入提示文本
    print_live=True,  # 开启实时打印
    collect_stats=True,  # 开启统计信息
)  # 函数调用完成

print(f'\n--- 返回的统计信息 ---')  # 打印统计标题
print(f'首 token 延迟: {result.get("first_token_time", 0):.3f} 秒')  # 打印首 token 延迟
print(f'总耗时: {result["total_time"]:.2f} 秒')  # 打印总耗时
print(f'生成 token 数: {result.get("token_count", 0)}')  # 打印 token 数
print(f'生成速度: {result.get("tokens_per_second", 0):.1f} tokens/秒')  # 打印生成速度

print('\n--- 测试静默模式（不实时打印） ---')  # 打印静默模式测试标题
result2 = stream_chat(  # 以静默模式调用工具函数
    prompt='中国最长的河流是什么？',  # 输入提示文本
    print_live=False,  # 关闭实时打印
    collect_stats=True,  # 开启统计信息
)  # 函数调用完成
print(f'静默模式完整回复: {result2["text"]}')  # 打印完整回复
print(f'总耗时: {result2["total_time"]:.2f} 秒')  # 打印总耗时

In [ ]:
# === 清理资源 ===
import os  # 导入操作系统模块
import signal  # 导入信号模块

try:  # 尝试终止服务进程
    server_process.terminate()  # 发送 SIGTERM 信号终止服务
    server_process.wait(timeout=10)  # 等待进程退出，最多 10 秒
    print(f'服务进程 (PID={server_process.pid}) 已终止')  # 打印终止成功信息
except NameError:  # 如果 server_process 变量未定义
    print('未找到服务进程变量，可能未启动或已在其他单元格中清理')  # 打印提示
except Exception as e:  # 其他异常
    print(f'终止进程时出错: {e}')  # 打印错误信息
    try:  # 尝试强制终止
        os.kill(server_process.pid, signal.SIGKILL)  # 发送 SIGKILL 强制终止
        print('已强制终止进程')  # 打印强制终止信息
    except Exception:  # 强制终止也失败
        print('请手动运行: kill -9 <PID> 终止进程')  # 提示手动终止

print('资源清理完成')  # 打印清理完成提示

## 常见问题排查

### 问题一：流式输出在收到第一个 token 后卡住

**现象：** 收到第一个 token 后，流就停止了，不再输出新内容。

**可能原因及解决方法：**
- **网络问题：** 检查客户端与服务端之间的网络连接是否稳定
- **服务过载：** 如果有大量并发请求，服务可能来不及处理。减少并发数或等待其他请求完成
- **显存不足：** 服务可能因 OOM 崩溃。使用 `nvidia-smi` 检查显存状态，必要时降低 `--mem-fraction-static`

### 问题二：收到空的 chunk（`delta.content` 为 None）

**现象：** 遍历流时，部分 chunk 的 `choices[0].delta.content` 为 `None`。

**解决方法：**
- 这是**正常现象**，流的第一个和最后一个 chunk 通常不含 content
- 在代码中添加判空过滤：`if chunk.choices[0].delta.content:` 即可
- 不要假设每个 chunk 都一定包含文本内容

---

**结语：** 流式输出是提升大模型交互体验的关键技术。通过本 Notebook，你已掌握使用 OpenAI SDK 和 requests 两种方式实现流式调用，以及如何测量首 token 延迟等性能指标。建议将 `stream_chat()` 工具函数集成到你的实际项目中。